# 12. The Z2 sector structure of the transfer matrix

The temporal entropy stops being usable at a time that falls sharply with frustration, and the
explanation on record was an exceptional point: two eigenvalues approach, their eigenvectors
coalesce. Notebook 11 found that account incomplete, so this notebook builds the transfer matrix
densely at small temporal size and inspects it directly. Every number here is exact, a property of
the operator rather than of a solver.

What it establishes: the next-nearest-neighbour coupling enlarges the temporal site and adds no
transpose symmetry, the eigenbasis conditioning worsens at p different from zero without the
leading pair being to blame, and an exact Z2 sign operator K splits the spectrum into two sectors
that hold the two eigenvalue families. The last section turns that symmetry into a repair: a power
method projected onto one sector, which removes the branch ambiguity from the eigenvalue route by
construction.

In [1]:
include("../src/thesislib.jl")
using LinearAlgebra, Printf, Logging, JLD2
ITensors.disable_warn_order()
Logging.disable_logging(Logging.Warn)

const DENSE_GUARD = 2500
const PS = (0.0, 0.1, 0.3, 0.5)

# contract a tMPO into a plain matrix; rows are the primed legs, columns the unprimed ones
function dense_mpo_matrix(mpo::MPO)
    contracted = mpo[1]
    for i in 2:length(mpo)
        contracted = contracted * mpo[i]
    end
    unprimed = [noprime(s) for s in inds(contracted) if plev(s) == 0]
    col = combiner(unprimed...)
    row = combiner(prime.(unprimed)...)
    return Matrix(row * contracted * col, combinedind(row), combinedind(col))
end

# the temporal chain, and the dense size it would need
function tmpo_at(p, scheme, Nt, dt)
    mpo, scaffold = build_tmpo(AlcarazParams(lambda=1.0, p=p), scheme, Nt * dt; dt=dt, nbeta=0)
    sites = siteinds(scaffold)
    return mpo, dim(sites[1]), length(sites), prod(dim.(sites))
end

# a diagonal sign pattern on the temporal site, tensored along the chain, is permutation invariant,
# so it does not matter in which order the combiner laid the sites out
function sign_operator(s, Nt)
    return Diagonal(kron([s for _ in 1:Nt]...))
end

# K commutes exactly when M has no element joining a plus state to a minus one, since
# [M,K]_ij = M_ij (k_j - k_i) and the signs are +-1. That is cheaper than forming the commutator.
function commutes_with(M, k)
    plus = findall(>(0), k)
    minus = findall(<(0), k)
    off = sum(abs2, view(M, plus, minus)) + sum(abs2, view(M, minus, plus))
    return 2 * sqrt(off) / norm(M) < 1e-10
end

# the one nontrivial sign pattern that commutes with the transfer matrix, found by search
function nontrivial_K(M, d, Nt)
    for bits in 1:(2^d - 1)
        s = [(bits >> (i - 1)) & 1 == 1 ? -1.0 : 1.0 for i in 1:d]
        s[1] == -1.0 && continue            # an overall sign is trivial
        k = kron([s for _ in 1:Nt]...)
        commutes_with(M, k) && return s, Diagonal(k)
    end
    return nothing, nothing
end

println("temporal site dimension of the Alcaraz tMPO")
@printf("%-6s %-8s %-8s\n", "p", "VD2", "WII")
for p in PS
    _, d_vd2, _, _ = tmpo_at(p, AlcarazVD2(), 2, 0.5)
    _, d_wii, _, _ = tmpo_at(p, AlcarazWII(), 2, 0.5)
    @printf("%-6.1f %-8d %-8d\n", p, d_vd2, d_wii)
end

temporal site dimension of the Alcaraz tMPO


p      VD2      WII     
0.0    3        2       


0.1    7        3       
0.3    7        3       
0.5    7        3       


## The sizes that can be reached, and the transpose

The dense dimension is the temporal site dimension raised to the number of temporal sites, and the
number of sites is T/dt. At the production step of dt=0.1 a dense matrix reaches T=0.3, far short
of the times where the method fails. Two moves buy the range back.

The first is a coarser step. A transfer matrix at dt=0.5 or 1.0 is a poorer approximation to the
same evolution, but it is built from the same network with the same structure, and the questions
asked below are structural: which operators commute with it, how its eigenvectors decompose, how
far its eigenbasis is from orthogonal. The second is the choice of kernel. VD2 is the production
one and stays closest to the physics; WII has the smaller temporal site and reaches twice the
number of sites, at the cost of degrading to first order for a next-nearest-neighbour coupling. VD2
is used for the checks that must be production-faithful and WII where reach matters, and each table
says which it used.

The first thing to measure is the transpose. If the transfer matrix equalled its own transpose its
left and right eigenvectors would be the same object and the rigidity would be one by construction.
The cell contracts the tMPO exactly and reports the relative distance to its transpose, together
with the dense size at each setting.

In [2]:
println("dense size and distance to the transpose\n")
@printf("%-6s %-5s %-5s %-4s %-8s %-14s\n", "kernel", "p", "dt", "Nt", "dim", "|M-M^T|/|M|")
for (name, scheme) in (("VD2", AlcarazVD2()), ("WII", AlcarazWII()))
    for dt in (0.5, 1.0), Nt in (3, 4)
        for p in PS
            mpo, d, nsites, dim_dense = tmpo_at(p, scheme, Nt, dt)
            if dim_dense > DENSE_GUARD
                @printf("%-6s %-5.1f %-5.1f %-4d %-8d %-14s\n", name, p, dt, Nt, dim_dense, "too big")
                continue
            end
            M = dense_mpo_matrix(mpo)
            @printf("%-6s %-5.1f %-5.1f %-4d %-8d %-14.2e\n", name, p, dt, Nt, dim_dense,
                    norm(M - transpose(M)) / norm(M))
            M = nothing
            GC.gc()
        end
    end
end

dense size and distance to the transpose



kernel p     dt    Nt   dim      |M-M^T|/|M|   
VD2    0.0   0.5   3    27       1.34e+00      


VD2    0.1   0.5   3    343      1.38e+00      
VD2    0.3   0.5   3    343      1.36e+00      


VD2    0.5   0.5   3    343      1.36e+00      
VD2    0.0   0.5   4    81       1.39e+00      


VD2    0.1   0.5   4    2401     1.40e+00      
VD2    0.3   0.5   4    2401     1.40e+00      


...

WII    0.5   0.5   4    81       1.31e+00      
WII    0.0   1.0   3    8        1.14e+00      


WII    0.1   1.0   3    27       1.30e+00      
WII    0.3   1.0   3    27       1.27e+00      


WII    0.5   1.0   3    27       1.25e+00      
WII    0.0   1.0   4    16       1.27e+00      


WII    0.1   1.0   4    81       1.36e+00      
WII    0.3   1.0   4    81       1.34e+00      


WII    0.5   1.0   4    81       1.33e+00      


The temporal site is larger at p != 0: three states against seven for VD2, two against three for WII.
The next-nearest-neighbour coupling adds a channel, the one that carries the memory of an unfinished
fermion line, so the transfer matrix at p != 0 is a bigger operator rather than a perturbed one.
This is also why the dense cost jumps: 7 to the power Nt against 3 to the power Nt.

The transpose gives a clean negative. The distance to the transpose is 1.3 to 1.4 at every setting,
and the square root of two is what two matrices with no relation to each other give, so there is no
sense in which this transfer matrix equals its own transpose. For scale, the symmetric Ising
construction gives 1e-18 and XXZ under VD2 gives 0.35.

The point is that p=0 is no better than p=0.3 here. The left and right eigenvectors are unrelated at
every coupling, including the one where the method reaches T=15, so the absence of a transpose
symmetry cannot be what limits the frustrated couplings. The absolute size should not be read too
closely: this asymmetry comes from the order the Trotter layers are assembled in and shrinks with
the step, and these matrices use dt=0.5 and 1.0 against a production step of 0.1. What matters is
that it does not depend on p.

## The spectrum and the conditioning of the eigenbasis

The claim being tested is that two eigenvalues approach each other and their eigenvectors coalesce.
Both halves are measurable here. The approach is the modulus ratio of the leading pair and their
separation in phase, and the coalescence is the conditioning of the eigenvector matrix: a basis of
independent directions has a condition number of order one, and a basis collapsing onto a common
direction has one that diverges.

The rigidity of an individual pair is also available exactly, since the left eigenvectors are the
right eigenvectors of the transpose. That is the same quantity the production runs estimate as
1/(||L|| ||R||) after bi-normalisation, computed here without any truncation.

The cell reports, for each coupling and each temporal size, the leading modulus, the ratio of the
second modulus to it, the phase of the second relative to the first, the exact rigidity of both,
and the condition number of the full eigenvector matrix. WII at dt=1.0, so a temporal size of Nt
corresponds to an evolution time of Nt.

In [3]:
function exact_rigidity(FR, FL, j)
    lam = FR.values[j]
    iL = argmin(abs.(FL.values .- lam))          # match the left partner by eigenvalue
    r = FR.vectors[:, j]
    l = FL.vectors[:, iL]
    return abs(transpose(l) * r) / (norm(l) * norm(r))
end

println("WII at dt=1.0, so Nt is the evolution time\n")
for p in PS
    @printf("p=%.1f\n", p)
    @printf("  %-4s %-7s %-9s %-11s %-10s %-10s %-10s %-10s\n",
            "Nt", "dim", "|mu0|", "|mu1/mu0|", "dphi1/pi", "r(mu0)", "r(mu1)", "cond(V)")
    for Nt in 3:7
        mpo, d, nsites, dim_dense = tmpo_at(p, AlcarazWII(), Nt, 1.0)
        dim_dense > DENSE_GUARD && continue
        M = dense_mpo_matrix(mpo)
        FR = eigen(M)
        FL = eigen(transpose(M))
        order = sortperm(abs.(FR.values), rev=true)
        i0, i1 = order[1], order[2]
        dphi = mod(angle(FR.values[i1]) - angle(FR.values[i0]) + pi, 2pi) - pi
        @printf("  %-4d %-7d %-9.4f %-11.5f %-10.3f %-10.2e %-10.2e %-10.2e\n",
                Nt, dim_dense, abs(FR.values[i0]), abs(FR.values[i1] / FR.values[i0]), dphi / pi,
                exact_rigidity(FR, FL, i0), exact_rigidity(FR, FL, i1), cond(FR.vectors))
        M = nothing; FR = nothing; FL = nothing
        GC.gc()
    end
    println()
end

WII at dt=1.0, so Nt is the evolution time

p=0.0
  Nt   dim     |mu0|     |mu1/mu0|   dphi1/pi   r(mu0)     r(mu1)     cond(V)   


  3    8       1.6571    0.99333     0.085      9.88e-01   9.89e-01   1.17e+00  
  4    16      2.1268    0.99637     0.063      9.73e-01   9.74e-01   1.30e+00  


  5    32      2.7142    0.99771     0.050      9.53e-01   9.54e-01   1.46e+00  
  6    64      3.4530    0.99842     0.042      9.28e-01   9.29e-01   1.66e+00  


  7    128     4.3853    0.99885     0.036      8.99e-01   8.99e-01   1.90e+00  



p=0.1
  Nt   dim     |mu0|     |mu1/mu0|   dphi1/pi   r(mu0)     r(mu1)     cond(V)   
  3    27      1.6117    0.99356     -0.932     6.96e-01   6.37e-01   2.46e+01  
  4    81      2.0263    0.99057     -0.946     5.98e-01   5.66e-01   9.36e+01  


...

  5    243     2.4985    0.99979     -0.862     3.89e-01   4.28e-01   2.70e+02  
  6    729     3.0790    0.99218     -0.774     2.97e-01   3.20e-01   1.39e+03  


  7    2187    3.9494    0.99312     -0.668     2.17e-01   2.72e-01   4.75e+03  



p=0.5
  Nt   dim     |mu0|     |mu1/mu0|   dphi1/pi   r(mu0)     r(mu1)     cond(V)   
  3    27      1.5824    0.94986     0.997      5.64e-01   6.41e-01   2.14e+01  
  4    81      2.0910    0.97026     -0.993     4.39e-01   4.17e-01   1.06e+02  


  5    243     2.4940    0.95230     -0.706     2.85e-01   2.77e-01   3.10e+02  
  6    729     3.2097    0.93695     -0.845     2.68e-01   1.39e-01   1.20e+03  


  7    2187    3.7464    0.99969     -0.452     7.44e-02   1.54e-01   5.83e+03  



The miniature reproduces the phenomenology. At p=0 the second eigenvalue sits at a small relative
phase, 0.085 of pi falling to 0.036, the boundary tower with its gap closing as 1/T. At every
p different from zero it sits near pi, between 0.67 and 0.997 of it: the displaced family, present
at every size and absent at p=0, exactly as the block data show.

The conditioning separates sharply. At p=0 the condition number of the eigenvector matrix runs
1.17 to 1.90 from Nt=3 to 7, so the eigenbasis stays a basis. At p=0.1 it runs 24.6 to 29900 over
the same sizes, and the exact rigidity of the leading pair falls from 0.696 to 0.309.

The reason is not the one on record. The stated mechanism is that two eigenvalues approach and
their eigenvectors rotate onto a common direction. Here the leading pair at p=0 is the closer of
the two, modulus ratio 0.9989 and phase gap 0.036 pi, and the basis stays conditioned at 1.9; at
p different from zero the partner is a phase of pi away, the largest separation available at equal
modulus, and the basis is ill-conditioned by three or four orders. Proximity of eigenvalues does
not drive it. Two cautions: the kernel here is WII, a structural probe that drops to first order
for a next-nearest-neighbour coupling, and equal Nt is not equal matrix size, which the next
section corrects.

## The sector operator, and which sector the displaced partner is in

The displaced-tower work established a diagonal operator K that commutes with the imaginary-time
transfer matrix and separates the two families by sign. It was never checked with the real-time
rows present, and its relation to the conditioning was never asked.

The cell searches for it rather than assuming it: every diagonal sign pattern on the temporal site
is tensored along the chain and tested for commutation, and the one that commutes is used. Because
the operator is the same factor on every site it is invariant under permutations of the sites, so
the order the combiner laid them out in does not matter.

Three things follow from it. Whether the eigenvectors decompose into sectors at all: for a matrix
commuting with K and with distinct eigenvalues every eigenvector must be a K eigenvector, so any
eigenvector that is not one marks a pair of eigenvalues from opposite sectors that have become
degenerate. Which sector holds the leading eigenvalue and which holds its pi-displaced partner: if
they differ, the two cannot mix, and removing the partner from the block is a legitimate operation
rather than a choice. And how much of the ill-conditioning survives the separation, which is the
condition number computed inside each sector.

In [4]:
const NB12_CACHE = "../results/data/nb12_sectors.jld2"
sectors = isfile(NB12_CACHE) ? load(NB12_CACHE, "sectors") : Dict{Any,Any}()

function sector_report(p, scheme, kernel_name, Nt, dt)
    key = (kernel_name, p, Nt)
    haskey(sectors, key) && return
    mpo, d, nsites, dim_dense = tmpo_at(p, scheme, Nt, dt)
    dim_dense > DENSE_GUARD && return
    @printf("diagonalising %s p=%.1f Nt=%d (dim %d) ...\n", kernel_name, p, Nt, dim_dense)
    flush(stdout)
    M = dense_mpo_matrix(mpo)
    s, K = nontrivial_K(M, d, nsites)
    F = eigen(M)
    charge = [real(dot(F.vectors[:, i], K * F.vectors[:, i])) /
              real(dot(F.vectors[:, i], F.vectors[:, i])) for i in eachindex(F.values)]
    plus = findall(x -> x > 0.9, charge)
    minus = findall(x -> x < -0.9, charge)
    i0 = argmax(abs.(F.values))
    # the partner: among members within a fifth of the leading modulus, the one nearest to pi
    near = [j for j in eachindex(F.values) if j != i0 && abs(F.values[j]) > 0.8 * abs(F.values[i0])]
    to_pi = [abs(abs(mod(angle(F.values[j]) - angle(F.values[i0]) + pi, 2pi) - pi) - pi) for j in near]
    ip = isempty(near) ? i0 : near[argmin(to_pi)]
    sectors[key] = (; dim=dim_dense, pattern=s, ntotal=length(charge),
                    nmixed=length(charge) - length(plus) - length(minus),
                    q0=charge[i0], qp=charge[ip],
                    dphi=mod(angle(F.values[ip]) - angle(F.values[i0]) + pi, 2pi) - pi,
                    cond_all=cond(F.vectors), cond_plus=cond(F.vectors[:, plus]),
                    cond_minus=cond(F.vectors[:, minus]))
    M = nothing; F = nothing
    GC.gc()
    jldsave(NB12_CACHE; sectors=sectors)
end

# p=0 is carried to larger Nt so the conditioning can be compared at matched dense dimension:
# the temporal site is smaller there, so equal Nt would compare a small matrix against a large one
for p in PS
    for Nt in (p == 0.0 ? (3:7) : (3:4))
        sector_report(p, AlcarazVD2(), "VD2", Nt, 0.5)
    end
    for Nt in 3:7
        sector_report(p, AlcarazWII(), "WII", Nt, 1.0)
    end
end

for kernel_name in ("VD2", "WII")
    @printf("\n%s\n", kernel_name)
    @printf("  %-5s %-4s %-7s %-14s %-8s %-11s %-9s %-10s %-10s %-8s\n",
            "p", "Nt", "dim", "not K eigvec", "K(mu0)", "K(partner)", "dphi/pi",
            "cond all", "cond sector", "gain")
    for p in PS, Nt in 3:7
        key = (kernel_name, p, Nt)
        haskey(sectors, key) || continue
        r = sectors[key]
        worst = max(r.cond_plus, r.cond_minus)
        @printf("  %-5.1f %-4d %-7d %-14s %-8.0f %-11.0f %-9.3f %-10.2e %-10.2e %-8.1f\n",
                p, Nt, r.dim, @sprintf("%d/%d", r.nmixed, r.ntotal), r.q0, r.qp, r.dphi / pi,
                r.cond_all, worst, r.cond_all / worst)
    end
end


VD2
  p     Nt   dim     not K eigvec   K(mu0)   K(partner)  dphi/pi   cond all   cond sector gain    


  0.0   3    27      0/27           1        -1          0.147     9.65e+00   9.65e+00   1.0     
  0.0   4    81      0/81           1        -1          0.335     2.83e+01   2.83e+01   1.0     


  0.0   5    243     0/243          1        1           0.355     5.01e+01   5.01e+01   1.0     
  0.0   6    729     0/729          1        1           0.465     1.08e+02   1.07e+02   1.0     
  0.0   7    2187    0/2187         1        -1          0.575     2.99e+02   2.95e+02   1.0     
  0.1   3    343     0/343          1        -1          -0.901    3.46e+02   3.46e+02   1.0     
  0.1   4    2401    0/2401         1        -1          -0.925    5.81e+03   5.81e+03   1.0     
  0.3   3    343     0/343          -1       1           0.974     2.95e+02   2.95e+02   1.0     
  0.3   4    2401    0/2401         -1       1           0.985     1.87e+03   1.87e+03   1.0     
  0.5   3    343     0/343          1        -1          -0.997    2.73e+02   2.73e+02   1.0     
  0.5   4    2401    0/2401         1        -1          -0.999    1.14e+03   1.14e+03   1.0     

WII
  p     Nt   dim     not K eigvec   K(mu0)   K(partner)  dphi/pi   cond all   cond sector gain    
  0.0   3    8

The sector operator exists with the real-time rows present. The search returns [1,-1,1] at p=0 and
[1,-1,1,1,-1,-1,1] at p different from zero for VD2, commuting to better than 1e-10; only the
imaginary-time version had been checked before. Under VD2 the decomposition is exact: not one
eigenvector out of 2401 fails to be a K eigenvector, at any coupling or size. Under WII about one
in six does, so WII breaks the sector structure the exact operator has — a second argument for VD2
at a next-nearest-neighbour coupling, independent of the norm-drift measurement in notebook 2.

The result that matters is the last two columns of the VD2 table: at every coupling with p
different from zero the leading eigenvalue and the member nearest a relative phase of pi carry
opposite K charge. The two are in different sectors of an exact symmetry, so they cannot mix, and
an exceptional point between them is forbidden rather than merely distant. Their near-degeneracy
in modulus is a crossing the symmetry enforces.

Compared at matched dimension the conditioning gap is real but modest: 50 against 273 to 346 near
dimension 300, and 299 against 1140 to 5810 near dimension 2400, so four to nineteen times worse
at a frustrated coupling, and not monotone in p — p=0.1 is the worst and p=0.5 the best.
Restricting to a sector does not help: the condition number of the worse sector equals that of the
whole basis in every VD2 row. The ill-conditioning lives inside a sector, so separating the
families fixes the selection problem and leaves the conditioning untouched. The next section does
the separating.

## Projecting the power method onto one sector

Because K is exact and diagonal, the temporal Hilbert space splits into two blocks that the
transfer matrix never connects, and the two eigenvalue families live one per block. Confining the
iteration to one block removes the cause of the post-selection failures rather than patching the
symptoms. The largest-modulus selector swapped family because the two families' modulus curves
cross; inside one sector there is one curve and nothing to swap to. The solver produced
non-converged rungs at the crossing because it had to resolve two families a few parts in ten
thousand apart; inside one sector the other family does not exist. The branch tracker had to guess
which returned number continued its branch; inside one sector the branch is the sector label,
fixed before the run, and a pi jump cannot occur. This is the temporal-network version of
conserving a quantum number in DMRG: a block-diagonal matrix is diagonalised block by block.

The projection is nearly free. Applying K to a temporal state multiplies components of each site
tensor by a sign, leaving the bond dimension unchanged, and the projector is half the sum or
difference of the state and its image. One subtlety matters: truncation does not preserve a global
sector, so a projected seed alone is not enough — a control run at T=4 drifted to the dominant
sector within a few hundred iterations. The projection is therefore applied after every
truncation, which `block_transfer_eigs` supports through a `project` keyword.

The cell runs a k=2 block in each sector at p=0.3 and production settings, and compares against
the unprojected k=4 block at the same times.

In [5]:
using LinearAlgebra

const KSECTOR_CACHE = "../results/data/nb11_ksector_pm.jld2"

# the sector operator: solve X W X = R W R^{-1} on a bulk tensor of the production tMPO
function sector_signs(p)
    mpo, _ = build_alcaraz_tmpo(3.0; p=p, lambda=1.0, dt=0.1, nbeta=4, MPO_alg="VD2")
    Tb = mpo[div(length(mpo), 2)]
    phys = [noprime(s) for s in inds(Tb) if plev(s) == 0 && hastags(s, "Site")]
    lnks = [s for s in inds(Tb) if hastags(s, "Link")]
    d = dim(phys[1])
    Warr = Array(Tb, lnks[1], lnks[2], prime(phys[1]), phys[1])
    sx = [0.0 1.0; 1.0 0.0]
    Wx = zeros(ComplexF64, size(Warr))
    for a in 1:2, b in 1:2, a2 in 1:2, b2 in 1:2
        Wx[a, b, :, :] .+= sx[a, a2] * sx[b, b2] * Warr[a2, b2, :, :]
    end
    rows = Matrix{ComplexF64}[]
    for a in 1:2, b in 1:2
        push!(rows, kron(Matrix(1.0I, d, d), Wx[a, b, :, :]) - kron(transpose(Warr[a, b, :, :]), Matrix(1.0I, d, d)))
    end
    ns = nullspace(vcat(rows...); rtol=1e-10)
    size(ns, 2) == 1 || error("intertwiner nullspace is $(size(ns,2))-dimensional")
    return sign.(real.(diag(reshape(ns[:, 1], d, d))))
end

function apply_K(psi::MPS, Rd)
    out = copy(psi)
    for i in 1:length(out)
        s = siteind(out, i)
        out[i] = noprime(out[i] * ITensor(collect(Diagonal(Rd)), s', s))
    end
    return out
end

project_sector(psi, Rd, sgn) = normalize(lincomb_mps([0.5, 0.5 * sgn], [psi, apply_K(psi, Rd)];
                                                     cutoff=1e-12, maxdim=128))

ksec = isfile(KSECTOR_CACHE) ? load(KSECTOR_CACHE, "res") : Dict{Tuple{Float64,Int},Any}()
Rd_03 = sector_signs(0.3)
println("R = diag", Int.(Rd_03))

for T in (2.0, 3.0, 4.0, 5.0, 6.0), sgn in (1, -1)
    haskey(ksec, (T, sgn)) && continue
    @printf("projected block p=0.3 T=%.1f sector %+d ...\n", T, sgn); flush(stdout)
    mpo, scaffold = build_alcaraz_tmpo(T; p=0.3, lambda=1.0, dt=0.1, nbeta=4, MPO_alg="VD2")
    sit = siteinds(scaffold)
    seed() = project_sector(normalize(complex.(randomMPS(sit, linkdims=4))), Rd_03, sgn)
    elapsed = @elapsed begin
        theta, L, Rv, info = block_transfer_eigs(mpo, scaffold;
            k=2, maxdim=64, maxdims=collect(2:2:64), cutoff=1e-12,
            cutoffs=[fill(1e-8, 40); 1e-10], itermax=4000, eps_conv=1e-6,
            stuck_after=400, trunc_mode=:rtm, n_track=2,
            seedL=MPS[seed() for _ in 1:2], seedR=MPS[seed() for _ in 1:2],
            project=psi -> project_sector(psi, Rd_03, sgn))
    end
    q = [real(overlap_noconj(L[j], apply_K(Rv[j], Rd_03)) / overlap_noconj(L[j], Rv[j])) for j in eachindex(theta)]
    ksec[(T, sgn)] = (; theta=collect(theta), q=q, reason=string(info[:reason]),
                      niters=info[:niters], elapsed=elapsed)
    jldsave(KSECTOR_CACHE; res=ksec)
end

fine_03 = Dict(k[2] => v for (k, v) in load("../results/data/cluster/sweep_rtm_eigs_p0.3_fine.jld2", "done")
               if k[1] == "rtm_eigs_p0.3_fine" && !haskey(v, :error))
println("\nper sector against the unprojected k=4 block at the same times\n")
@printf("%5s %7s | %-30s | %-30s\n", "T", "", "sector +1", "sector -1")
for T in (2.0, 3.0, 4.0, 5.0, 6.0)
    line = @sprintf("%5.1f block:", T)
    if haskey(fine_03, T)
        line *= " " * join([@sprintf("%.4f", abs(t)) for t in fine_03[T].theta], " ")
    end
    println(line)
    for sgn in (1, -1)
        haskey(ksec, (T, sgn)) || continue
        r = ksec[(T, sgn)]
        @printf("      sector %+d: |th|=%s  K=%s  %s@%d  %.0fs\n", sgn,
                join([@sprintf("%.4f", abs(t)) for t in r.theta], " "),
                join([@sprintf("%+.3f", real(x)) for x in r.q], " "), r.reason, r.niters, r.elapsed)
    end
end

R = diag

[1, -1, 1, 1, -1, -1, 1]

per sector against the unprojected k=4 block at the same times



    T         | sector +1                      | sector -1                     
  2.0 block: 1.7115 1.6514 1.3887 1.1781


      sector +1: |th|=1.6511 1.3012  K=+1.000 +1.000  converged@28  10s


      sector -1: |th|=1.7118 1.3886  K=-1.000 -1.000  converged@30  4s
  3.0 block: 1.7165 1.6506 1.6332 1.5806
      sector +1: |th|=1.6481 1.6332  K=+1.000 +1.000  converged@59  14s
      sector -1: |th|=1.7165 1.5811  K=-1.000 -1.000  converged@77  21s
  4.0 block: 1.7074 1.7072 1.6057 0.7133
      sector +1: |th|=1.7072 1.1328  K=+1.000 +1.000  stuck@431  256s
      sector -1: |th|=1.7074 1.6161  K=-1.000 -1.000  converged@192  89s
  5.0 block: 1.7153 1.6966 1.6717 1.6180
      sector +1: |th|=1.7153 1.6692  K=+1.000 +1.000  converged@157  133s
      sector -1: |th|=1.6964 1.6752  K=-1.000 -1.000  stuck@620  498s
  6.0 block: 1.7133 1.7046 1.6906 1.6669
      sector +1: |th|=1.7133 1.6796  K=+1.000 +1.000  converged@291  337s
      sector -1: |th|=1.7041 1.6892  K=-1.000 -1.000  stuck@1252  1370s


The projection holds: the sector charge of every converged pair is one to three decimals through
several hundred truncated iterations, and the two k=2 sector blocks together reproduce the
unprojected k=4 block, member by member. The one disagreement sits on the worst-converged bottom
member at T=2, 1.3012 against 1.1781, and is left open.

The crossing that broke every mixed method is resolved. The minus sector leads at T=2 to 4 with
1.7118, 1.7165, 1.7074 and the plus sector from T=5 with 1.7153, 1.7133; the mixed ladder's
largest-modulus sequence is the upper envelope of these two curves, and its pi-sized phase steps
were the envelope switching family. At T=4, where the families sit two parts in ten thousand apart
and the unprojected block returned a garbage member at 0.7133, each sector converges cleanly.

The costs are honest. The dominant sector converges in 28 to 291 iterations. The subdominant
sector past the crossing reports stuck at 620 and 1252, because its own in-sector gap is tight
there (1.6964 against 1.6752 at T=5); the purity holds regardless, and notebook 11 showed a stuck
iteration still carries an accurate eigenvalue, but those rungs cost twenty minutes locally and
the projection roughly doubles a sweep. What the sector ladder buys the eigenvalue route at
p >= 0.3 is a ladder per family with no branch ambiguity; what it does not buy is conditioning,
which the previous section showed to be intra-sector. The production step is a cluster mode
running the eigenvalue sweeps sector by sector.

## Summary

The next-nearest-neighbour coupling changes the transfer matrix structurally: the temporal site
grows from three states to seven because the sigma-z memory channel appears, and that channel
carries a second eigenvalue family displaced by pi. There is no transpose symmetry at any
coupling, so left and right eigenvectors are unrelated everywhere, and that cannot be what
separates p=0 from the frustrated points.

The exact sign operator K splits the spectrum into two sectors, one per family, with the leading
eigenvalue and its pi-displaced partner always on opposite sides. An exceptional point between
them is therefore forbidden, and their modulus near-degeneracy is a symmetry-enforced crossing.
What genuinely worsens with frustration is the conditioning inside each sector, four to nineteen
times at matched size, largest at p=0.1 rather than growing with p; no projection improves it.

The sector-projected power method, applied after every truncation, resolves the two families into
separate ladders through their crossing at p=0.3 and removes the branch ambiguity from the
eigenvalue route by construction. Its cluster version is the production step; the conditioning of
the eigenvector route remains the open limit.